# Progressive Student Model Training

Trains a split pipeline for distilling our Claude-based extraction pipeline into local models:
- **NER**: BioBERT-base (110M) with BIOES tagging + sliding window
- **RE**: BioBERT-base (110M) pair classifier with typed entity markers

Trained at progressive scales: 1 → 10 → 100 → 400 BioRED gold examples.

**Runtime**: Select GPU (Runtime → Change runtime type → T4 GPU)

## Setup

In [ ]:
# Mount Google Drive (optional — for persisting results across sessions)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone the repo and install dependencies (torch is pre-installed on Colab)
!git clone https://github.com/Currie32/biolink-hub.git /content/biolink-hub
%cd /content/biolink-hub
!pip install -e '.[train]' -q

In [ ]:
# Verify GPU
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(name)s %(levelname)s %(message)s",
)

## Quick smoke test (N=1, N=10)

Run the split pipeline at the two smallest scales to validate everything works.
Expected time: ~10 min on T4 (BioBERT-base is 110M params).

In [ ]:
from bioextract.model.train_progressive import run_progressive

results_small = run_progressive(scales=[1, 10])

### Smoke test checks

- **N=1**: The pipeline should memorize the single example
- **N=10**: NER should tag at least some entities on unseen text; RE should predict a few relationships

In [ ]:
import json
print(json.dumps(results_small, indent=2))

## Full experiment (N=100, N=400)

Run the larger scales. Expected time: ~2-3 hr on T4 (BioBERT-base is 110M params).

You can run these individually if you're worried about session timeouts.

In [ ]:
results_large = run_progressive(scales=[100, 400])

## Review results

In [ ]:
# Load the full results file (includes all scales run so far)
import json
from pathlib import Path

results_path = Path("models/progressive_results.json")
with open(results_path) as f:
    all_results = json.load(f)

print(json.dumps(all_results, indent=2))

In [ ]:
from bioextract.model.train_progressive import _print_summary
_print_summary(all_results)

## Save student model

Saves the best trained model to Google Drive and creates a zip for local download.
The model with the highest N is used as the student model.

In [ ]:
import json
import shutil
from pathlib import Path

# Find the best N from results
results_path = Path("models/progressive_results.json")
with open(results_path) as f:
    all_results = json.load(f)

best_n = max(
    (v["n"] for v in all_results.values() if "n" in v),
    default=None,
)
if best_n is None:
    raise RuntimeError("No trained models found in results.")

ner_src = Path(f"models/ner_n{best_n}")
re_src = Path(f"models/re_n{best_n}")
print(f"Best model: N={best_n}")
print(f"  NER: {ner_src} (exists: {ner_src.exists()})")
print(f"  RE:  {re_src} (exists: {re_src.exists()})")

# --- Save to Google Drive ---
drive_dest = Path("/content/drive/MyDrive/biolink-hub/student_model")
drive_dest.mkdir(parents=True, exist_ok=True)

shutil.copy(results_path, drive_dest / "progressive_results.json")
shutil.copytree(ner_src, drive_dest / "ner", dirs_exist_ok=True)
shutil.copytree(re_src, drive_dest / "re", dirs_exist_ok=True)
print(f"\nSaved to Drive: {drive_dest}")

# --- Create zip for local download ---
zip_path = Path("/content/student_model.zip")
shutil.make_archive(str(zip_path.with_suffix("")), "zip", "/content/drive/MyDrive/biolink-hub/student_model")
print(f"Zip created: {zip_path} ({zip_path.stat().st_size / 1e6:.0f} MB)")

In [ ]:
# Download the zip to your local machine
from google.colab import files
files.download("/content/student_model.zip")